# Module 1: Apache Iceberg Fundamentals

This notebook contains code examples for Module 1 videos.

## Setup

Initialize Spark session for Module 1 examples.

In [1]:
# Initialize Spark session for working with Iceberg tables

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module1_Fundamentals") \
    .getOrCreate()

ModuleNotFoundError: No module named 'pyspark'

## Video: The Open Lakehouse

In [ ]:
# Display Spark configuration showing how to connect to Catalog (Polaris) and Storage (MinIO)
# Inside the spark-defaults.conf file, we have 
import os

spark_conf_path = os.path.expanduser('~/.sparkconf/spark-defaults.conf')

with open(spark_conf_path, 'r') as f:
    spark_config = f.read()
    print(spark_config)

## Video: Modeling Data into an Apache Iceberg Table

In [ ]:
# Create an unpartitioned Iceberg table from the NYC taxi Parquet files
# This demonstrates the basic table structure without partitioning

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# Query the first 3 columns to verify the table was created successfully

spark.sql("""
    SELECT 
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    FROM polaris.taxi.trips_unpartitioned
    LIMIT 10
""").show()

### View Table Files in MinIO

[Open trips_unpartitioned in MinIO](http://localhost:9001/browser/warehouse/taxi/trips_unpartitioned/)

Login: `admin` / `password`

In [ ]:
# Use the .files metadata table to inspect data files and aggregated table statistics

spark.sql("""
    SELECT 
        SUBSTRING(file_path, POSITION('/data/' IN file_path)) as file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

spark.sql("""
    SELECT 
        COUNT(*) as total_files,
        SUM(record_count) as total_records,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_unpartitioned.files
""").show()

## Video: Hidden Partitioning in Apache Iceberg


In [ ]:
# Create a table with hidden partitioning using the days() transform
# Iceberg automatically partitions by day without requiring a separate day column

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

In [ ]:
# Query with timestamp filter - Iceberg automatically applies partition pruning
# Check Spark UI (http://localhost:4040/SQL/) to see which files were scanned vs. skipped

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""").show()

### View Query Execution in Spark UI

[Open Spark SQL Tab](http://localhost:4040/SQL/)

Check the query metrics to see:
- **Number of result data files**: Only files for the queried date range
- **Number of skipped data files**: Files from other partitions that were pruned
- **Partition pruning**: Iceberg automatically applied the partition filter

In [ ]:
# View column metrics (min/max bounds) stored in manifest files for predicate pushdown
# These statistics enable file skipping without opening the actual data files

spark.sql("""
    SELECT 
        SUBSTRING(file_path, POSITION('day=' IN file_path)) as file_path,
        partition,
        record_count,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        readable_metrics.PULocationID.lower_bound as PULocationID_min,
        readable_metrics.PULocationID.upper_bound as PULocationID_max,
        readable_metrics.total_amount.lower_bound as total_amount_min,
        readable_metrics.total_amount.upper_bound as total_amount_max
    FROM polaris.taxi.trips_by_day.files
    ORDER BY partition DESC
    LIMIT 10
""").show(truncate=False)